In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from heuristic_feature_selector import HeuristicFeatureSelector

In [2]:
# 1) Load a toy dataset (all numeric, no encoding needed)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# 2) Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [3]:
# 3) Instantiate the heuristic selector
hfs = HeuristicFeatureSelector(
    # max_features=15,      # e.g. cap at 15 features
    random_state=42,
    verbose=True          
)

In [4]:
# 4) Run heuristic FS on the training set
result = hfs.fit_transform(X_train, y_train)

[HeuristicFS] Meta-profile: {'n_samples': 455, 'n_features': 30, 'p_to_n_ratio': 0.06593406593406594, 'task_type': 'binary', 'class_imbalance': 0.6263736263736264, 'mean_abs_correlation': 0.3964349862073119, 'mean_variance': 14243.351417174741, 'sparsity': 0.004835164835164835}
[HeuristicFS] Final k=30 for FS
[HeuristicFS] Chosen strategy: wrapper / rfe_logreg with params {'n_features_to_select': 30, 'step': 0.1}
[HeuristicFS] Selected 30 / 30 features in 0.037 seconds.


In [5]:
# 5) Inspect what it did
print("\n=== Meta-profile ===")
for k, v in result.meta_profile.items():
    print(f"{k}: {v}")

print("\nFS family:", result.fs_family)
print("FS method:", result.fs_method)
print("Selected features:", result.selected_features)
print(f"Features: {result.n_features_before} -> {result.n_features_after}")
print(f"FS runtime: {result.fs_runtime:.4f} seconds")


=== Meta-profile ===
n_samples: 455
n_features: 30
p_to_n_ratio: 0.06593406593406594
task_type: binary
class_imbalance: 0.6263736263736264
mean_abs_correlation: 0.3964349862073119
mean_variance: 14243.351417174741
sparsity: 0.004835164835164835

FS family: wrapper
FS method: rfe_logreg
Selected features: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']
Features: 30 -> 30
FS runtime: 0.0374 seconds


In [6]:
# 6) Transform the test set with the same feature subset
X_train_fs = result.X_selected
X_test_fs = hfs.transform(X_test)


In [7]:
# 7) Train a normal model on the reduced features
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_fs, y_train)
y_pred = clf.predict(X_test_fs)

print("\nTest accuracy with heuristic FS:", accuracy_score(y_test, y_pred))


Test accuracy with heuristic FS: 0.956140350877193


In [8]:

# Baseline: all 30 features
rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)
base_acc = accuracy_score(y_test, y_pred_base)
print("Baseline accuracy (no FS):", base_acc)

# With heuristic FS (you already have this)
rf_fs = RandomForestClassifier(random_state=42)
rf_fs.fit(X_train_fs, y_train)
y_pred_fs = rf_fs.predict(X_test_fs)
fs_acc = accuracy_score(y_test, y_pred_fs)
print("Heuristic FS accuracy:", fs_acc)


Baseline accuracy (no FS): 0.956140350877193
Heuristic FS accuracy: 0.956140350877193
